# Orbit Wars — Expert Iteration (ExIt) Training on Colab

GPU-accelerated Expert Iteration training on Google Colab with persistent storage via Google Drive.

**How ExIt works (vs PPO):**
- PPO: collect rollouts → compute advantages (shared across all planets in a step) → policy gradient update
- ExIt: collect games → for each per-planet decision, run forward simulation to score candidates → supervised learning on improved targets

**Key advantages:**
- Per-decision credit assignment (each planet gets its own quality estimate from search)
- Data reuse across iterations (accumulated dataset)
- Orbit prediction (intercept calculation for orbiting planets)

**Pipeline:**
1. Setup (install deps, mount Drive, clone repo)
2. GPU verification
3. Configure experiment
4. **Train** (optional BC pretrain → ExIt loop: collect → search → train → eval)
5. Generate submission
6. Evaluate against baselines
7. TensorBoard monitoring

In [ ]:
#@title 1. Setup: Mount Drive, Clone Repo, Install Dependencies

import os, sys

# ── Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Persistent output directory on Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/orbit_wars_outputs'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/submissions', exist_ok=True)
print(f'Drive output dir: {DRIVE_OUTPUT}')

# ── Clone or update repo ────────────────────────────────────────────────────
REPO_URL = 'https://github.com/oshbocker/orbit_wars.git'  # <-- EDIT THIS
REPO_DIR = '/content/orbit_wars'

if os.path.exists(REPO_DIR):
    print(f'Repo already cloned at {REPO_DIR}, pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working dir: {os.getcwd()}')

# ── Install dependencies ────────────────────────────────────────────────────
os.system('pip install -q --upgrade "kaggle-environments>=1.28.0"')
os.system('pip install -q "stable-baselines3[extra]>=2.3" gymnasium pyyaml tensorboard')

print('\nSetup complete.')

In [ ]:
#@title 2. GPU Verification

import torch

print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    DEVICE = 'cuda'
else:
    print('WARNING: No GPU detected! Training will be slow.')
    print('Go to Runtime > Change runtime type > GPU')
    DEVICE = 'cpu'

print(f'\nUsing device: {DEVICE}')

## Configuration

Edit the cell below to configure the Expert Iteration experiment.

The ExIt pipeline runs:
1. **BC pretraining** (optional) — clone apex behavior via supervised learning
2. **ExIt loop** — for each iteration:
   - Play games with current policy, recording all per-planet decisions
   - For each decision, run forward simulation to score candidate actions
   - Train on search-improved targets via supervised learning
   - Periodically evaluate against apex/random

In [ ]:
#@title 3. Experiment Config

from src.config import load_train_config

# Load ExIt config
cfg = load_train_config('configs/expert_iteration.yaml')

# ── Colab GPU overrides ──────────────────────────────────────────────────────
cfg.device = DEVICE

# ExIt parameters — adjust based on your GPU and time budget
cfg.exit.iterations = 200         # total ExIt iterations
cfg.exit.games_per_iter = 30      # games to collect per iteration
cfg.exit.search_depth = 15        # forward simulation steps per candidate
cfg.exit.search_candidates = 10   # targets to evaluate per planet
cfg.exit.train_epochs = 5         # supervised training epochs per iteration
cfg.exit.train_batch_size = 256
cfg.exit.train_lr = 0.001
cfg.exit.lr_schedule = 'cosine'
cfg.exit.dataset_max_iters = 10   # keep data from last N iterations

# BC pretraining (optional, helps bootstrap the policy)
cfg.imitation.enabled = True
cfg.imitation.bc_expert = 'apex'
cfg.imitation.bc_games = 100
cfg.imitation.bc_epochs = 30

# Evaluation
cfg.eval.eval_every = 10
cfg.eval.eval_games = 10

# Point outputs to Google Drive for persistence
cfg.save_dir = f'{DRIVE_OUTPUT}/checkpoints'
cfg.log_dir = f'{DRIVE_OUTPUT}/logs'

# Checkpointing
cfg.checkpoint_every = 10

print(f'Run: {cfg.run_name}')
print(f'Device: {cfg.device}')
print(f'ExIt iterations: {cfg.exit.iterations}')
print(f'Games/iter: {cfg.exit.games_per_iter}')
print(f'Search: depth={cfg.exit.search_depth}, candidates={cfg.exit.search_candidates}, '
      f'fractions={cfg.exit.search_fractions}')
print(f'Training: epochs={cfg.exit.train_epochs}, batch={cfg.exit.train_batch_size}, '
      f'lr={cfg.exit.train_lr}, schedule={cfg.exit.lr_schedule}')
print(f'Dataset: keep last {cfg.exit.dataset_max_iters} iterations')
print(f'BC pretrain: enabled={cfg.imitation.enabled}, expert={cfg.imitation.bc_expert}, '
      f'games={cfg.imitation.bc_games}, epochs={cfg.imitation.bc_epochs}')
print(f'Eval: every {cfg.eval.eval_every} iters, {cfg.eval.eval_games} games vs {cfg.eval.eval_opponents}')
print(f'\nCheckpoints: {cfg.save_dir}/{cfg.run_name}')
print(f'Logs: {cfg.log_dir}/{cfg.run_name}')

In [ ]:
#@title 4. Train (Expert Iteration)

import yaml, sys, os
sys.path.insert(0, '/content/orbit_wars')
os.chdir('/content/orbit_wars')

# Serialize config to temp file for the training script
temp_cfg = '/tmp/exit_train_cfg.yaml'
with open(temp_cfg, 'w') as f:
    yaml.dump({
        'seed': cfg.seed,
        'run_name': cfg.run_name,
        'device': cfg.device,
        'save_dir': cfg.save_dir,
        'log_dir': cfg.log_dir,
        'checkpoint_every': cfg.checkpoint_every,
        'log_every': cfg.log_every,
        'env': {
            'max_targets': cfg.env.max_targets,
            'k_neighbors': cfg.env.k_neighbors,
            'ship_fractions': cfg.env.ship_fractions,
        },
        'model': {
            'embed_dim': cfg.model.embed_dim,
            'n_heads': cfg.model.n_heads,
            'n_layers': cfg.model.n_layers,
            'ff_dim': cfg.model.ff_dim,
            'pos_hidden': cfg.model.pos_hidden,
        },
        'exit': {
            'iterations': cfg.exit.iterations,
            'games_per_iter': cfg.exit.games_per_iter,
            'search_depth': cfg.exit.search_depth,
            'search_candidates': cfg.exit.search_candidates,
            'search_fractions': cfg.exit.search_fractions,
            'search_temperature': cfg.exit.search_temperature,
            'train_epochs': cfg.exit.train_epochs,
            'train_batch_size': cfg.exit.train_batch_size,
            'train_lr': cfg.exit.train_lr,
            'lr_warmup_steps': cfg.exit.lr_warmup_steps,
            'lr_schedule': cfg.exit.lr_schedule,
            'dataset_max_iters': cfg.exit.dataset_max_iters,
            'champion_pool_size': cfg.exit.champion_pool_size,
            'value_loss_coef': cfg.exit.value_loss_coef,
            'entropy_coef': cfg.exit.entropy_coef,
            'max_grad_norm': cfg.exit.max_grad_norm,
            'opponent': cfg.exit.opponent,
            'four_player_prob': cfg.exit.four_player_prob,
        },
        'imitation': {
            'enabled': cfg.imitation.enabled,
            'bc_expert': cfg.imitation.bc_expert,
            'bc_games': cfg.imitation.bc_games,
            'bc_demo_opponent': cfg.imitation.bc_demo_opponent,
            'bc_epochs': cfg.imitation.bc_epochs,
            'bc_lr': cfg.imitation.bc_lr,
            'bc_batch_size': cfg.imitation.bc_batch_size,
            'bc_skip_steps': cfg.imitation.bc_skip_steps,
        },
        'eval': {
            'eval_every': cfg.eval.eval_every,
            'eval_games': cfg.eval.eval_games,
            'eval_opponents': cfg.eval.eval_opponents,
        },
        # PPO section not used by ExIt but needed for config compat
        'ppo': {
            'total_updates': 0,
            'lr': cfg.exit.train_lr,
        },
    }, f)

print(f'Config written to {temp_cfg}')
print(f'Starting Expert Iteration training...\n')

!python -m src.exit_train --config {temp_cfg}

CHECKPOINT_DIR = f'{cfg.save_dir}/{cfg.run_name}'
print(f'\nCheckpoints saved to: {CHECKPOINT_DIR}')

## Submission

Generate and submit to Kaggle. The apex agent is a reliable baseline submission.
The trained ExIt checkpoint is saved to Drive for future submission generation.

In [ ]:
#@title 5. Generate Submission
import shutil
from pathlib import Path
from agents.rl_agent import export_submission

# ── Save apex submission (reliable baseline) ────────────────────────────────
submission_path = f'{DRIVE_OUTPUT}/submissions/submission_apex.py'
export_submission(None, output_path=submission_path, mode='apex')
print(f'Apex submission: {submission_path}')

# ── Save hybrid submission ──────────────────────────────────────────────────
hybrid_path = f'{DRIVE_OUTPUT}/submissions/submission_hybrid.py'
export_submission(None, output_path=hybrid_path, mode='hybrid')
print(f'Hybrid submission: {hybrid_path}')

# ── Copy trained checkpoint to submissions dir ──────────────────────────────
ckpt_src = Path(CHECKPOINT_DIR) / 'ckpt_last.pt'
if ckpt_src.exists():
    ckpt_dst = f'{DRIVE_OUTPUT}/submissions/ckpt_exit_last.pt'
    shutil.copy2(ckpt_src, ckpt_dst)
    print(f'Trained checkpoint copied to: {ckpt_dst}')
else:
    print(f'No checkpoint found at {ckpt_src}')

print(f'\nTo upload apex submission:')
print(f'  kaggle competitions submit orbit-wars -f "{submission_path}" -m "Apex agent"')
print(f'\nTo upload hybrid submission:')
print(f'  kaggle competitions submit orbit-wars -f "{hybrid_path}" -m "Hybrid agent"')

## Evaluation

Run the trained ExIt policy head-to-head against apex and random agents.

In [ ]:
#@title 6. Evaluate Trained Model

import torch
from pathlib import Path
from src.config import load_train_config
from src.policy import TransformerPolicy
from src.logging import make_eval_agent
from evaluation.evaluate import run_games, print_results

# Load checkpoint
ckpt_path = Path(CHECKPOINT_DIR) / 'ckpt_last.pt'
if not ckpt_path.exists():
    print(f'No checkpoint found at {ckpt_path}')
    print('Available checkpoints:')
    for p in sorted(Path(DRIVE_OUTPUT, 'checkpoints').rglob('ckpt_last.pt')):
        print(f'  {p}')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    policy = TransformerPolicy(cfg.model, cfg.env).to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    policy.load_state_dict(ckpt['policy'])
    policy.eval()
    print(f'Loaded checkpoint: {ckpt_path} (iteration {ckpt["update"]})')

    eval_agent = make_eval_agent(policy, cfg, device)

    N_GAMES = 20

    from agents.apex import agent as apex_agent
    print(f'\nExIt vs Apex ({N_GAMES} games):')
    r = run_games(eval_agent, apex_agent, n_games=N_GAMES, verbose=True)
    print_results('exit', 'apex', r)

    from kaggle_environments.envs.orbit_wars.orbit_wars import random_agent
    print(f'\nExIt vs Random ({N_GAMES} games):')
    r2 = run_games(eval_agent, random_agent, n_games=N_GAMES, verbose=True)
    print_results('exit', 'random', r2)

## Monitoring

TensorBoard shows train/loss, train/policy_loss, train/value_loss, train/win_rate,
eval/apex_win_rate, eval/random_win_rate, and more.

**Note:** TensorBoard may block execution of cells below it — run this last.

In [ ]:
#@title 7. TensorBoard

%load_ext tensorboard
%tensorboard --logdir {DRIVE_OUTPUT}/logs